# 02 Random Forest：用企鹅数据理解“很多棵树投票”

这次我们用 `penguins.csv`。每一行是一只企鹅，里面有嘴巴长度、嘴巴深度、脚蹼长度、体重等信息。

目标是预测企鹅属于哪一种：

- Adelie
- Gentoo
- Chinstrap

Random Forest 的直觉很像“找一群同学投票”：每棵决策树都根据企鹅身体数据猜一次，最后看哪种企鹅得到最多票。


## 运行前说明

这些 notebook 是教学演示，不是医学诊断或生物学结论。我们会使用你放在项目里的真实表格数据，但这里只是学习机器学习模型怎么工作。

数据路径：

- 心脏病数据：`../data/ml_data/heart_data.csv`
- 企鹅数据：`../data/ml_data/penguins.csv`

推荐在本项目已有的 Docker/Jupyter 环境里运行，因为 `environment/Dockerfile` 已经安装了 `numpy`、`pandas`、`scikit-learn`、`matplotlib`、`seaborn`。

如果你想在本机 Python 里运行，可以先安装：

```bash
pip install numpy pandas scikit-learn matplotlib seaborn notebook
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. 读取并认识企鹅数据

先看前几行，确认有哪些列。这里 `species` 是我们要预测的答案。


In [ ]:
data_path = Path('../data/ml_data/penguins.csv')
penguins = pd.read_csv(data_path)

# 第一列只是原始行号，没有实际预测意义，这里删掉。
if '' in penguins.columns:
    penguins = penguins.drop(columns=[''])

print('数据形状：', penguins.shape)
penguins.head()


In [ ]:
print('企鹅种类数量：')
print(penguins['species'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=penguins, x='species', hue='species', palette='Set2', legend=False)
plt.title('三种企鹅的样本数量')
plt.xlabel('企鹅种类')
plt.ylabel('数量')
plt.show()


## 2. 清理缺失值

有些企鹅的身体数据缺失，比如嘴巴长度或体重是 `NA`。为了让例子简单，我们直接删除这些缺失行。


In [ ]:
features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
target = 'species'

penguins_clean = penguins.dropna(subset=features + [target]).copy()

print('清理前行数：', len(penguins))
print('清理后行数：', len(penguins_clean))
penguins_clean[features + [target]].head()


## 3. 用图感受不同企鹅的差异

下面只画两个指标：嘴巴长度和脚蹼长度。

如果不同颜色分得比较开，说明这些身体指标确实能帮助我们猜企鹅种类。


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=penguins_clean,
    x='bill_length_mm',
    y='flipper_length_mm',
    hue='species',
    palette='Set2',
    s=75,
)
plt.title('嘴巴长度和脚蹼长度可以帮助区分企鹅种类')
plt.xlabel('嘴巴长度 bill_length_mm')
plt.ylabel('脚蹼长度 flipper_length_mm')
plt.show()


## 4. 训练 Random Forest

一棵决策树会问很多“是/否”问题，例如：

- 脚蹼长度是否大于 205 mm？
- 嘴巴深度是否小于 16 mm？
- 体重是否大于 4500 g？

Random Forest 会训练很多棵这样的树，然后让它们投票。


In [ ]:
X = penguins_clean[features]
y = penguins_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

model = RandomForestClassifier(
    n_estimators=120,
    max_depth=4,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)
print('训练完成')
print('测试集准确率：', round(model.score(X_test, y_test), 3))


## 5. 预测一只新企鹅

我们输入一只企鹅的身体数据，让森林里的树一起投票。`predict_proba` 会显示每个种类得到的“支持比例”。


In [ ]:
new_penguin = pd.DataFrame({
    'bill_length_mm': [46.0],
    'bill_depth_mm': [14.5],
    'flipper_length_mm': [215.0],
    'body_mass_g': [5000.0],
})

predicted_species = model.predict(new_penguin)[0]
probabilities = pd.Series(model.predict_proba(new_penguin)[0], index=model.classes_).sort_values(ascending=False)

display(new_penguin)
print('预测企鹅种类：', predicted_species)
print('每个种类的预测概率：')
display(probabilities.to_frame('probability'))


## 6. 评估：哪些企鹅容易被猜错？

混淆矩阵的行是真实种类，列是预测种类。对角线越深，说明猜对得越多。


In [ ]:
y_pred = model.predict(X_test)

print('准确率：', round(accuracy_score(y_test, y_pred), 3))
print()
print('分类报告：')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap='Greens')
plt.title('Random Forest 企鹅种类混淆矩阵')
plt.show()


## 7. 看哪些身体指标最有用

Random Forest 可以给出“特征重要性”。它表示森林里的树在做判断时，经常依赖哪些指标。


In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(x=importance.values, y=importance.index, hue=importance.index, palette='Set2', legend=False)
plt.title('Random Forest 认为最有用的企鹅身体指标')
plt.xlabel('重要性分数')
plt.ylabel('指标')
plt.show()

importance


## 8. 小结

Random Forest 适合：

- 特征之间关系比较复杂，不一定能用一条直线讲清楚。
- 想让很多棵树一起投票，减少单棵树的偶然错误。
- 想知道哪些特征大概更重要。

在企鹅例子里，它会综合嘴巴、脚蹼、体重等信息，投票猜企鹅种类。
